# 03 - Entrenamiento LoRA

Fine-tuning del modelo base con LoRA/QLoRA.

**Modelo base:** Qwen/Qwen2.5-Coder-1.5B-Instruct

**Configuraciones:**
- Config A: rank=8, alpha=32, targets=[q_proj, v_proj]
- Config B: rank=32, alpha=64, targets=[q_proj, v_proj, k_proj, o_proj]

**Hipótesis:**
- Config A: rank bajo es suficiente para patrones SQL básicos (fácil y media)
- Config B: rank alto captura patrones complejos (subconsultas, ventanas, WITH)

In [ ]:
import json
import os
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset, Dataset
import pandas as pd
import matplotlib.pyplot as plt

## 1. Configuración

In [ ]:
# Config
with open('config.json', 'r') as f:
    config = json.load(f)

MODELO_BASE = config['modelo_base']
SEMILLA = config['semilla']

# Hyperparámetros
BATCH_SIZE = 4
GRAD_ACCUM = 4
LR = 2e-4
EPOCHS = 3
MAX_SEQ_LEN = 2048

print(f'Modelo: {MODELO_BASE}')
print(f'Semilla: {SEMILLA}')
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 2. Preparar datos

In [ ]:
def construir_prompt(esquema, pregunta):
    return f"""### Esquema de la base de datos:
{esquema.strip()}

### Pregunta:
{pregunta.strip()}

### Consulta MySQL:
"""

def preparar_dataset(ruta_dataset='datos/entrenamiento/dataset_completo.json'):
    """Convierte el dataset en formato para entrenamiento."""
    with open(ruta_dataset, 'r') as f:
        data = json.load(f)
    
    # Para cada par, necesitamos el esquema correspondiente
    # Cargar esquemas
    esquemas = {}
    for archivo in Path('datos/esquemas/entrenamiento/').glob('*.sql'):
        esquemas[archivo.stem] = archivo.read_text()
    
    registros = []
    for item in data:
        # Asignar esquema aleatorio (para generalización)
        esquema = random.choice(list(esquemas.values()))
        prompt = construir_prompt(esquema, item['pregunta'])
        completacion = item['sql']
        texto = prompt + completacion
        registros.append({'text': texto})
    
    dataset = Dataset.from_list(registros)
    print(f'Dataset: {len(dataset)} ejemplos')
    return dataset

## 3. Configuración A: rank=8

In [ ]:
# Config A
lora_config_a = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)

print('=== Config A ===')
print(f'Rank: {lora_config_a.r}')
print(f'Alpha: {lora_config_a.lora_alpha}')
print(f'Target modules: {lora_config_a.target_modules}')

# Contar parámetros entrenables
def contar_parametros(modelo):
    total = sum(p.numel() for p in modelo.parameters())
    entrenables = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
    return total, entrenables

## 4. Entrenamiento Config A

In [ ]:
import random

def entrenar(config_lora, nombre_run, dataset):
    """Entrena el modelo con la configuración LoRA dada."""
    print(f'\n{"="*60}')
    print(f'Entrenando: {nombre_run}')
    print(f'{"="*60}')
    
    # Cargar modelo base
    tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    model = AutoModelForCausalLM.from_pretrained(
        MODELO_BASE,
        torch_dtype=torch.float16,
        device_map='auto',
        trust_remote_code=True
    )
    
    # Aplicar LoRA
    model = get_peft_model(model, config_lora)
    total, entrenables = contar_parametros(model)
    print(f'Parámetros totales: {total:,}')
    print(f'Parámetros entrenables: {entrenables:,} ({entrenables/total*100:.2f}%)')
    
    # Training arguments
    output_dir = f'resultados/training_{nombre_run}'
    training_args = SFTConfig(
        output_dir=output_dir,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        fp16=True,
        logging_steps=10,
        save_strategy='epoch',
        save_total_limit=2,
        seed=SEMILLA,
        max_seq_length=MAX_SEQ_LEN,
        report_to='none'
    )
    
    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        tokenizer=tokenizer
    )
    
    # Entrenar
    train_result = trainer.train()
    
    # Guardar adapter
    adapter_dir = f'adapter/{nombre_run}'
    os.makedirs(adapter_dir, exist_ok=True)
    trainer.model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    
    print(f'Adapter guardado en: {adapter_dir}')
    
    # Métricas
    metrics = train_result.metrics
    print(f'Tiempo entrenamiento: {metrics.get("train_runtime", 0):.1f}s')
    print(f'Loss final: {metrics.get("train_loss", 0):.4f}')
    
    return trainer, metrics

In [ ]:
# Preparar dataset
dataset = preparar_dataset()

# Entrenar Config A
trainer_a, metrics_a = entrenar(lora_config_a, 'config_a', dataset)

## 5. Configuración B: rank=32

In [ ]:
# Config B
lora_config_b = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)

print('=== Config B ===')
print(f'Rank: {lora_config_b.r}')
print(f'Alpha: {lora_config_b.lora_alpha}')
print(f'Target modules: {lora_config_b.target_modules}')

# Entrenar Config B
trainer_b, metrics_b = entrenar(lora_config_b, 'config_b', dataset)

## 6. Comparar configuraciones

In [ ]:
print('\n=== Comparación de configuraciones ===')
print(f'Config A - Loss: {metrics_a.get("train_loss", "N/A")}')
print(f'Config B - Loss: {metrics_b.get("train_loss", "N/A")}')

# Guardar comparación
comparacion = {
    'config_a': {
        'rank': 8,
        'alpha': 32,
        'targets': ['q_proj', 'v_proj'],
        'loss': metrics_a.get('train_loss'),
        'tiempo': metrics_a.get('train_runtime'),
        'hipotesis': 'Rank bajo es suficiente para patrones SQL básicos'
    },
    'config_b': {
        'rank': 32,
        'alpha': 64,
        'targets': ['q_proj', 'v_proj', 'k_proj', 'o_proj'],
        'loss': metrics_b.get('train_loss'),
        'tiempo': metrics_b.get('train_runtime'),
        'hipotesis': 'Rank alto captura patrones SQL complejos (subconsultas, ventanas)'
    }
}

with open('resultados/comparacion_configs.json', 'w') as f:
    json.dump(comparacion, f, indent=2)

print('Comparación guardada')

## 7. Curvas de loss

In [ ]:
def plot_loss_curves(log_dir_a, log_dir_b):
    """Grafica las curvas de loss de ambas configuraciones."""
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # TODO: Cargar logs de entrenamiento y graficar
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss')
    ax.set_title('Curvas de pérdida: Config A vs Config B')
    ax.legend()
    plt.tight_layout()
    plt.savefig('resultados/curvas_loss.png', dpi=150)
    plt.show()

print('Las curvas de loss se graficarán al completar el entrenamiento')